In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("udfApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/25 11:29:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/25 11:29:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
cabsDF = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "./Files/Cabs.csv",
)
cabsDF.createOrReplaceTempView("Cabs")
cabsDF.show()

+---------+--------------------+--------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------+--------------------+---------------+
|CabNumber|VehicleLicenseNumber|                Name|     LicenseType|Active|PermitLicenseNumber| VehicleVinNumber|WheelchairAccessible|VehicleYear|VehicleType|TelephoneNumber|             Website|             Address|LastDateUpdated|
+---------+--------------------+--------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------+--------------------+---------------+
| T802127C|              C19641|          ABCON INC.|OWNER MUST DRIVE|   YES|               NULL|5TDBK3EH0DS268018|                NULL|       2016|       NULL|  (718)438-1100|                NULL|41-24   38 STREET...|     04/22/2020|
| T525963C|             5362996| ACCEPTABLE TAXI LLC|    NAM

25/05/25 11:29:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [14]:
# Function to convert first character of each word to uppercase
def capitalize_first_letter(s):
    if s is None:
        return None
    return ', '.join(word.capitalize() for word in s.split(","))
# Registering the UDF
capitalize_udf = udf(capitalize_first_letter, StringType())
# Using the UDF to transform the 'Name' column
cabsDF.withColumn("Name_Case", capitalize_udf(col("Name"))).show(truncate=False)

+---------+--------------------+------------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------------+--------------------------------------------+---------------+------------------------+
|CabNumber|VehicleLicenseNumber|Name                    |LicenseType     |Active|PermitLicenseNumber|VehicleVinNumber |WheelchairAccessible|VehicleYear|VehicleType|TelephoneNumber|Website                   |Address                                     |LastDateUpdated|Name_Case               |
+---------+--------------------+------------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------------+--------------------------------------------+---------------+------------------------+
|T802127C |C19641              |ABCON INC.              |OWNER MUST DRIVE|YES   |NULL               |5TDBK3EH0DS268018

In [15]:
spark.udf.register("capitalize_first_letter_sql", capitalize_first_letter, StringType())
# Using the UDF in SQL
spark.sql("""
    SELECT Name, capitalize_first_letter_sql(Name) AS Name_Case
    FROM Cabs
""").show(truncate=False)

+------------------------+------------------------+
|Name                    |Name_Case               |
+------------------------+------------------------+
|ABCON INC.              |Abcon inc.              |
|ACCEPTABLE TAXI LLC     |Acceptable taxi llc     |
|ALLIS CAB CORP          |Allis cab corp          |
|BENE CAB CORP           |Bene cab corp           |
|BOULOS TAXI CORP.       |Boulos taxi corp.       |
|CACERES,JAIME,P         |Caceres, Jaime, P       |
|CALCIUM ONE SERVICE INC.|Calcium one service inc.|
|CHARLES,WILBERT         |Charles, Wilbert        |
|CHAWKI,MICHAEL          |Chawki, Michael         |
|CHRYSOVALANTOU CORP,    |Chrysovalantou corp,    |
|COFI BOAT CORP.         |Cofi boat corp.         |
|DEKEL TAXI CAB CORP     |Dekel taxi cab corp     |
|FLORIAN & ROBERT INC    |Florian & robert inc    |
|GART CAB CORP           |Gart cab corp           |
|GAUTHIER,JACQUES        |Gauthier, Jacques       |
|GEORGAKOPOULOS, GEORGIA |Georgakopoulos,  georgia|
|GUJAR CAB C